In [1]:
from functions import calc_isp_sl
from utils import to_si, g0
import utils
import cantera as ct
from pint import UnitRegistry
ureg = UnitRegistry()
Q_ = ureg.Quantity

In [2]:
o_f_ratio = 6.0
temperature_h2 = Q_(20.270, 'K')
temperature_o2 = Q_(90.170, 'K')
pressure_chamber = Q_(3000, 'psi')

species_dict = utils.species_list_to_dict(ct.Species.list_from_file(utils.reactant_files[0]))
fuel_spec = [species_dict["H2(L)"]]
ox_spec = [species_dict["O2(L)"]]

oxidizer = ct.Solution(thermo="IdealGas", species = ox_spec)
oxidizer.TP = to_si(temperature_o2), to_si(pressure_chamber)

In [3]:
temperature_rp1 = Q_(293.15, 'K')
species_dict2 = utils.species_list_to_dict(ct.Species.list_from_file(utils.reactant_files[2]))
fuel_spec = [species_dict2["C6H6(L)"]]
fuel = ct.Solution(thermo='IdealGas', species=fuel_spec)
fuel.TP = to_si(temperature_rp1), to_si(pressure_chamber)
fuel()
fuel2 = utils.copy_solution(fuel)
oxidizer2 = utils.copy_solution(oxidizer)


       temperature   293.15 K
          pressure   2.0684e+07 Pa
           density   662.9 kg/m^3
  mean mol. weight   78.114 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy        6.1964e+05        4.8403e+07  J
   internal energy        5.8844e+05        4.5965e+07  J
           entropy            1624.9        1.2693e+05  J/K
    Gibbs function        1.4331e+05        1.1194e+07  J
 heat capacity c_p            1726.8        1.3489e+05  J/K
 heat capacity c_v            1620.4        1.2657e+05  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
           C6H6(L)                 1                 1            4.5928



In [4]:
oxidizer()


       temperature   90.17 K
          pressure   2.0684e+07 Pa
           density   882.81 kg/m^3
  mean mol. weight   31.998 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy       -4.0562e+05       -1.2979e+07  J
   internal energy       -4.2905e+05       -1.3729e+07  J
           entropy           -1382.1            -44223  J/K
    Gibbs function         -2.81e+05       -8.9914e+06  J
 heat capacity c_p                 0                 0  J/K
 heat capacity c_v           -259.84           -8314.5  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
             O2(L)                 1                 1           -11.993



In [5]:
o_f_ratio = 2.6
isp = calc_isp_sl(fuel, oxidizer, pressure_chamber, o_f_ratio, 15.0)[0]
print(f"ISP: {isp} s, {isp*g0} m/s")

Molar ratio: 6.347159197449841
Moles ox: 0.8638929723549349
CHAMBER CONDITIONS: 

       temperature   4027.3 K
          pressure   2.0684e+07 Pa
           density   16.771 kg/m^3
  mean mol. weight   27.151 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy       -1.2082e+05       -3.2805e+06  J
   internal energy       -1.3541e+06       -3.6765e+07  J
           entropy            9813.8        2.6645e+05  J/K
    Gibbs function       -3.9644e+07       -1.0764e+09  J
 heat capacity c_p            1747.8             47455  J/K
 heat capacity c_v            1441.6             39140  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
                 C         5.025e-09        1.1359e-08           -14.797
                CH        1.8831e-09         3.927e-09           -24.771

In [6]:
isp_frozen = calc_isp_sl(fuel2, oxidizer2, pressure_chamber, o_f_ratio, 3.0, frozen = True)[0]
print(f"Frozen ISP: {isp_frozen} s, {isp_frozen*g0} m/s")

Molar ratio: 6.347159197449841
Moles ox: 0.8638929723549349
CHAMBER CONDITIONS: 

       temperature   4027.3 K
          pressure   2.0684e+07 Pa
           density   16.771 kg/m^3
  mean mol. weight   27.151 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy       -1.2082e+05       -3.2805e+06  J
   internal energy       -1.3541e+06       -3.6765e+07  J
           entropy            9813.8        2.6645e+05  J/K
    Gibbs function       -3.9644e+07       -1.0764e+09  J
 heat capacity c_p            1747.8             47455  J/K
 heat capacity c_v            1441.6             39140  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
                 C         5.025e-09        1.1359e-08           -14.797
                CH        1.8831e-09         3.927e-09           -24.771

In [7]:
gas = ct.Solution(thermo="IdealGas", species=fuel_spec+ox_spec)
mixture_fraction = 1/o_f_ratio/(o_f_ratio+1)
print(mixture_fraction)
gas.set_mixture_fraction(mixture_fraction, fuel="C6H6(L):1",oxidizer="O2(L):1" )
gas()
gas.equivalence_ratio()

0.10683760683760682

       temperature   0.001 K
          pressure   0.00010644 Pa
           density   0.00043721 kg/m^3
  mean mol. weight   34.152 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy       -3.8128e+05       -1.3022e+07  J
   internal energy       -3.8128e+05       -1.3022e+07  J
           entropy           -2695.8            -92068  J/K
    Gibbs function       -3.8128e+05       -1.3022e+07  J
 heat capacity c_p            724.02             24727  J/K
 heat capacity c_v            480.57             16413  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
           C6H6(L)           0.10684           0.04671       -1.6701e+06
             O2(L)           0.89316           0.95329        -1.561e+06



0.36749289132445734

In [8]:
gas.set_equivalence_ratio(1.18163,fuel="C6H6(L):1",oxidizer="O2(L):1")
gas.TP = to_si(temperature_rp1), to_si(pressure_chamber) 
gas()


       temperature   293.15 K
          pressure   2.0684e+07 Pa
           density   324.81 kg/m^3
  mean mol. weight   38.275 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy       -1.2082e+05       -4.6245e+06  J
   internal energy       -1.8451e+05       -7.0619e+06  J
           entropy           -460.37            -17621  J/K
    Gibbs function             14133        5.4095e+05  J
 heat capacity c_p            479.67             18359  J/K
 heat capacity c_v            262.43             10045  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
           C6H6(L)           0.27778           0.13611            2.5984
             O2(L)           0.72222           0.86389          -0.15248



In [9]:
gas.equilibrate("HP")
gas_chamber = utils.extract_reaction_species(fuel, oxidizer)
fuel.TP = to_si(temperature_rp1), to_si(pressure_chamber)
oxidizer.TP = to_si(temperature_o2), to_si(pressure_chamber)
mixture = ct.Mixture([(fuel, 0.13611), (oxidizer,0.86389), (gas_chamber, 0)])
mixture()

************ Phase  ************
Moles:  0.13611

       temperature   293.15 K
          pressure   2.0684e+07 Pa
           density   662.9 kg/m^3
  mean mol. weight   78.114 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy        6.1964e+05        4.8403e+07  J
   internal energy        5.8844e+05        4.5965e+07  J
           entropy            1624.9        1.2693e+05  J/K
    Gibbs function        1.4331e+05        1.1194e+07  J
 heat capacity c_p            1726.8        1.3489e+05  J/K
 heat capacity c_v            1620.4        1.2657e+05  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
           C6H6(L)                 1                 1            4.5928

************ Phase  ************
Moles:  0.86389

       temperature   293.15 K
          pressure   2.0

In [10]:
gas_chamber()


       temperature   293.15 K
          pressure   2.0684e+07 Pa
           density   101.93 kg/m^3
  mean mol. weight   12.011 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy         5.966e+07        7.1657e+08  J
   internal energy        5.9457e+07        7.1413e+08  J
           entropy            9451.8        1.1353e+05  J/K
    Gibbs function        5.6889e+07        6.8329e+08  J
 heat capacity c_p            1735.1             20841  J/K
 heat capacity c_v            1042.9             12526  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
                 C                 1                 1            280.34
     [ +110 minor]                 0                 0  

